# sheet_operate — Phase 5 匯出（merge LoRA → GGUF q4_K_M）
把 GRPO 訓完的 adapter 合併回 base model，量化成 GGUF 存到 Drive。
下載到本機後即可用 Ollama（`ollama create`）或 llama-cpp-python 直接推論。


In [ ]:
CFG = dict(
    drive_root = "/content/drive/MyDrive/公司/AI ML Research/未命名資料夾/sheet_operat",
    adapter    = "adapter_grpo_v1",     # 要匯出的 adapter（Drive 相對路徑）
    base_model = "Qwen/Qwen3-4B-Instruct-2507",
    quant      = "q4_k_m",              # 4070 8GB 的甜蜜點（約 2.5GB）
    lora_r = 64, lora_alpha = 64,
)

In [ ]:
%%capture
!pip install unsloth

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CFG["base_model"], max_seq_length = 8192,
    dtype = None, load_in_4bit = False,
)
model = FastLanguageModel.get_peft_model(
    model, r = CFG["lora_r"], lora_alpha = CFG["lora_alpha"], lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)

from safetensors.torch import load_file
from peft.utils import set_peft_model_state_dict
import glob

# 自動搜尋 adapter：先試 CFG 路徑，再掃同層兄弟資料夾（防資料夾名打字差異）
def find_adapter():
    p = os.path.join(CFG["drive_root"], CFG["adapter"], "adapter_model.safetensors")
    if os.path.exists(p):
        return p
    parent = os.path.dirname(CFG["drive_root"].rstrip("/"))
    for root in sorted(glob.glob(os.path.join(parent, "*"))):
        p = os.path.join(root, CFG["adapter"], "adapter_model.safetensors")
        if os.path.exists(p):
            CFG["drive_root"] = root      # 輸出的 gguf 也存到 adapter 所在資料夾
            return p
    raise AssertionError(f"{parent} 底下找不到 {CFG['adapter']}；該層內容：{os.listdir(parent)}")

adapter_path = find_adapter()
print("使用 adapter：", adapter_path)
set_peft_model_state_dict(model, load_file(adapter_path))

In [ ]:
# 合併 + 轉 GGUF + 量化（unsloth 一條龍）
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method=CFG["quant"])

import glob, shutil
gguf = sorted(glob.glob("gguf_out/*.gguf"), key=os.path.getsize)[-1]
dst = os.path.join(CFG["drive_root"], f"sheetops-{CFG['quant']}.gguf")
shutil.copy(gguf, dst)
print(f"完成：{dst}（{os.path.getsize(dst)/1e9:.2f} GB）")

## 下載後的本機步驟（Windows）
1. 從 Drive 下載 `sheetops-q4_k_m.gguf` 到 repo 的 `deploy/` 資料夾
2. `ollama create sheetops -f deploy/Modelfile`
3. 開始用：`python scripts/sheetops_cli.py 報表.xlsx "你的指令"`
   （或不裝 Ollama：`--gguf deploy/sheetops-q4_k_m.gguf` 直連推論）
